# Predicting Student Health Risk — Local Ensemble Baseline

A local-ready baseline derived from the supplied stacked tree notebook. It uses stratified validation, balanced accuracy, missing-value indicators, lightweight health features, and a probability blend of CatBoost, XGBoost, and LightGBM.

Start with `QUICK_RUN = True`; switch it off for the full training set and final submission.

In [1]:
from pathlib import Path
import time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
QUICK_RUN = False
QUICK_ROWS = 150_000
DATA_DIR = Path('.')

In [2]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

TARGET, ID_COL = 'health_condition', 'id'
LABELS = ['unhealthy', 'at-risk', 'fit']
LABEL_TO_INT = {label: i for i, label in enumerate(LABELS)}
INT_TO_LABEL = {i: label for label, i in LABEL_TO_INT.items()}

if QUICK_RUN and len(train) > QUICK_ROWS:
    train, _ = train_test_split(train, train_size=QUICK_ROWS, stratify=train[TARGET], random_state=RANDOM_STATE)
    train = train.reset_index(drop=True)

print(f'Training rows: {len(train):,} | Test rows: {len(test):,}')
display(train[TARGET].value_counts(normalize=True).rename('share').to_frame())

Training rows: 690,088 | Test rows: 295,753


,share
health_condition,
at-risk,0.858675
unhealthy,0.083647
fit,0.057678


## Feature engineering

In [3]:
def add_features(df):
    df = df.copy()
    eps = 1e-6
    base = [c for c in df.columns if c not in [ID_COL, TARGET]]
    df['missing_count'] = df[base].isna().sum(axis=1).astype('int8')
    df['bmi_distance_normal'] = (df['bmi'] - 22.0).abs()
    df['sleep_distance_8h'] = (df['sleep_duration'] - 8.0).abs()
    df['steps_per_exercise_min'] = df['step_count'] / (df['exercise_duration'] + eps)
    df['calories_per_step'] = df['calorie_expenditure'] / (df['step_count'] + eps)
    df['water_per_calorie'] = df['water_intake'] / (df['calorie_expenditure'] + eps)
    return df.replace([np.inf, -np.inf], np.nan)

train_fe = add_features(train)
test_fe = add_features(test)
FEATURES = [c for c in test_fe.columns if c != ID_COL]
CAT_COLS = test_fe[FEATURES].select_dtypes(exclude=np.number).columns.tolist()
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]
print(f'{len(NUM_COLS)} numerical + {len(CAT_COLS)} categorical features')

13 numerical + 6 categorical features


## Stratified holdout

In [4]:
train_idx, valid_idx = train_test_split(
    np.arange(len(train_fe)), test_size=0.20,
    stratify=train_fe[TARGET], random_state=RANDOM_STATE
)
X_train = train_fe.loc[train_idx, FEATURES].copy()
X_valid = train_fe.loc[valid_idx, FEATURES].copy()
y_train = train_fe.loc[train_idx, TARGET].map(LABEL_TO_INT)
y_valid = train_fe.loc[valid_idx, TARGET].map(LABEL_TO_INT)

## CatBoost

In [5]:
def catboost_frame(df):
    out = df.copy()
    for col in CAT_COLS:
        out[col] = out[col].fillna('<MISSING>').astype(str)
    return out

X_train_cat = catboost_frame(X_train)
X_valid_cat = catboost_frame(X_valid)
test_cat = catboost_frame(test_fe[FEATURES])

cat_model = CatBoostClassifier(
    iterations=1200 if not QUICK_RUN else 500,
    learning_rate=0.05, depth=8, loss_function='MultiClass',
    eval_metric='MultiClass', auto_class_weights='Balanced',
    random_seed=RANDOM_STATE, verbose=100, allow_writing_files=False,
)
cat_model.fit(X_train_cat, y_train, cat_features=CAT_COLS,
              eval_set=(X_valid_cat, y_valid), early_stopping_rounds=100)
cat_valid = cat_model.predict_proba(X_valid_cat)
cat_test = cat_model.predict_proba(test_cat)
print('CatBoost balanced accuracy:', balanced_accuracy_score(y_valid, cat_valid.argmax(1)))

0:	learn: 1.0225094	test: 1.0226127	best: 1.0226127 (0)	total: 377ms	remaining: 7m 31s


100:	learn: 0.1803179	test: 0.1827076	best: 0.1827076 (100)	total: 34.2s	remaining: 6m 12s


200:	learn: 0.1706717	test: 0.1767394	best: 0.1767394 (200)	total: 1m 9s	remaining: 5m 47s


300:	learn: 0.1650031	test: 0.1752139	best: 0.1752139 (300)	total: 1m 47s	remaining: 5m 21s


400:	learn: 0.1597053	test: 0.1742884	best: 0.1742725 (397)	total: 2m 26s	remaining: 4m 51s


500:	learn: 0.1547994	test: 0.1740049	best: 0.1739589 (479)	total: 3m 3s	remaining: 4m 16s


600:	learn: 0.1502378	test: 0.1738352	best: 0.1737617 (586)	total: 3m 40s	remaining: 3m 39s


Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.1737616512
bestIteration = 586

Shrink model to first 587 iterations.


CatBoost balanced accuracy: 0.9499152949926414


## Shared encoding for XGBoost and LightGBM

In [6]:
preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median', add_indicator=True), NUM_COLS),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1,
                                   encoded_missing_value=-1, dtype=np.float32)),
    ]), CAT_COLS),
], verbose_feature_names_out=False)

X_train_enc = preprocessor.fit_transform(X_train)
X_valid_enc = preprocessor.transform(X_valid)
test_enc = preprocessor.transform(test_fe[FEATURES])
sample_weight = y_train.map((1 / y_train.value_counts(normalize=True)).to_dict()).to_numpy()
print(X_train_enc.shape)

(552070, 31)


## XGBoost

In [7]:
xgb_model = XGBClassifier(
    objective='multi:softprob', num_class=3, eval_metric='mlogloss',
    n_estimators=1000 if not QUICK_RUN else 400, learning_rate=0.05,
    max_depth=8, min_child_weight=15, subsample=.85, colsample_bytree=.8,
    reg_lambda=5, tree_method='hist', random_state=RANDOM_STATE, n_jobs=-1,
)
xgb_model.fit(X_train_enc, y_train, sample_weight=sample_weight,
              eval_set=[(X_valid_enc, y_valid)], verbose=100)
xgb_valid = xgb_model.predict_proba(X_valid_enc)
xgb_test = xgb_model.predict_proba(test_enc)
print('XGBoost balanced accuracy:', balanced_accuracy_score(y_valid, xgb_valid.argmax(1)))

[0]	validation_0-mlogloss:1.05088


[100]	validation_0-mlogloss:0.21923


[200]	validation_0-mlogloss:0.19838


[300]	validation_0-mlogloss:0.18892


[400]	validation_0-mlogloss:0.18049


...

[700]	validation_0-mlogloss:0.15941


[800]	validation_0-mlogloss:0.15383


[900]	validation_0-mlogloss:0.14887


[999]	validation_0-mlogloss:0.14452


XGBoost balanced accuracy: 0.9321879912855286


## LightGBM

In [8]:
lgb_model = LGBMClassifier(
    objective='multiclass', num_class=3,
    n_estimators=1200 if not QUICK_RUN else 500, learning_rate=0.04,
    num_leaves=63, max_depth=-1, min_child_samples=50,
    subsample=.85, colsample_bytree=.8, reg_lambda=5,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1,
)
lgb_model.fit(X_train_enc, y_train)
lgb_valid = lgb_model.predict_proba(X_valid_enc)
lgb_test = lgb_model.predict_proba(test_enc)
print('LightGBM balanced accuracy:', balanced_accuracy_score(y_valid, lgb_valid.argmax(1)))

LightGBM balanced accuracy: 0.9373706004939808


## Select blend weights on validation data

In [9]:
valid_probs = [cat_valid, xgb_valid, lgb_valid]
test_probs = [cat_test, xgb_test, lgb_test]
model_names = ['CatBoost', 'XGBoost', 'LightGBM']

candidates = []
grid = np.arange(0, 1.01, 0.1)
for w_cat in grid:
    for w_xgb in grid:
        w_lgb = 1 - w_cat - w_xgb
        if w_lgb < -1e-9: continue
        weights = np.array([w_cat, w_xgb, max(0, w_lgb)])
        blend = sum(w*p for w, p in zip(weights, valid_probs))
        score = balanced_accuracy_score(y_valid, blend.argmax(1))
        candidates.append((score, *weights))

best_score, *best_weights = max(candidates)
best_weights = np.array(best_weights)
print('Best balanced accuracy:', round(best_score, 6))
print(dict(zip(model_names, best_weights.round(2))))

valid_blend = sum(w*p for w, p in zip(best_weights, valid_probs))
valid_pred = valid_blend.argmax(1)
print(classification_report(y_valid, valid_pred, target_names=LABELS, digits=4))
display(pd.DataFrame(confusion_matrix(y_valid, valid_pred), index=LABELS, columns=LABELS))

Best balanced accuracy: 0.950137
{'CatBoost': np.float64(0.8), 'XGBoost': np.float64(0.1), 'LightGBM': np.float64(0.1)}
              precision    recall  f1-score   support

   unhealthy     0.6998    0.9635    0.8108     11545
     at-risk     0.9935    0.9383    0.9651    118512
         fit     0.7408    0.9486    0.8319      7961

    accuracy                         0.9410    138018
   macro avg     0.8113    0.9501    0.8692    138018
weighted avg     0.9543    0.9410    0.9445    138018



,unhealthy,at-risk,fit
unhealthy,11124,373,48
at-risk,4723,111194,2595
fit,49,360,7552


## Create submission

In [10]:
test_blend = sum(w*p for w, p in zip(best_weights, test_probs))
submission = sample_submission.copy()
submission[TARGET] = pd.Series(test_blend.argmax(1)).map(INT_TO_LABEL)
assert submission.shape == sample_submission.shape
assert submission[TARGET].isin(LABELS).all()
assert submission[ID_COL].equals(test[ID_COL])

submission.to_csv('submission_local_ensemble.csv', index=False)
display(submission[TARGET].value_counts(normalize=True).rename('share').to_frame())
display(submission.head())
print('Saved submission_local_ensemble.csv')

,share
health_condition,
at-risk,0.813365
unhealthy,0.113318
fit,0.073318


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


Saved submission_local_ensemble.csv


## Next improvements

- Set `QUICK_RUN = False` and rerun on all rows.
- Replace the holdout with 3–5 fold out-of-fold predictions for more reliable blending.
- Tune decision thresholds or class-specific probability multipliers directly for balanced accuracy.
- Validate feature engineering with ablation tests; do not assume every ratio helps.
- Compare against each single model and keep the ensemble only if the gain is repeatable across folds.